In [14]:
# Import libraries
import pandas as pd
import numpy as np 
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split 

print("All libraries imported")

All libraries imported


In [16]:
#Load the real churn dataset
df = pd.read_csv('churn-bigml-80.csv')

print("Dataset loaded successfully")
print("Shape:", df.shape)
print()
print(df.head())

Dataset loaded successfully
Shape: (2666, 20)

  State  Account length  Area code International plan Voice mail plan  \
0    KS             128        415                 No             Yes   
1    OH             107        415                 No             Yes   
2    NJ             137        415                 No              No   
3    OH              84        408                Yes              No   
4    OK              75        415                Yes              No   

   Number vmail messages  Total day minutes  Total day calls  \
0                     25              265.1              110   
1                     26              161.6              123   
2                      0              243.4              114   
3                      0              299.4               71   
4                      0              166.7              113   

   Total day charge  Total eve minutes  Total eve calls  Total eve charge  \
0             45.07              197.4              

In [17]:
#Add missing values - randonly remove 5% of values from 2 columns to simulate this
np.random.seed(42)

#Add missing values to Account length and total day minutes 
df.loc[np.random.choice(df.index, 100, replace=False), 'Account length'] = np.nan
df.loc[np.random.choice(df.index, 100, replace=False), 'Total day minutes'] = np.nan

print("Missing values after introducing them:")
print(df.isnull().sum())

Missing values after introducing them:
State                       0
Account length            100
Area code                   0
International plan          0
Voice mail plan             0
Number vmail messages       0
Total day minutes         100
Total day calls             0
Total day charge            0
Total eve minutes           0
Total eve calls             0
Total eve charge            0
Total night minutes         0
Total night calls           0
Total night charge          0
Total intl minutes          0
Total intl calls            0
Total intl charge           0
Customer service calls      0
Churn                       0
dtype: int64


In [18]:
#Fill numerical cells with the mean of that column instead of deleting rows and losing data 

df['Account length'] = df['Account length'].fillna(df['Account length'].mean())
df['Total day minutes'] = df['Total day minutes'].fillna(df['Total day minutes'].mean())

# Confirm all missing values are gone
print("Missing values after fixing:")
print(df.isnull().sum().sum(), "missing values remaining")
print()
print("Account length mean used:", round(df['Account length'].mean(), 2))
print("Total day minutes mean used:", round(df['Total day minutes'].mean(), 2))

Missing values after fixing:
0 missing values remaining

Account length mean used: 100.85
Total day minutes mean used: 179.38


In [19]:
#Convert text columns to numbers
#Machines cant read words like Yes, No, True, False
#Swap them out for 0s and 1s

le = LabelEncoder()

# Yes becomes 1, No becomes 0
df['International plan'] = le.fit_transform(df['International plan'])
df['Voice mail plan'] = le.fit_transform(df['Voice mail plan'])

# True becomes 1, False becomes 0
df['Churn'] = le.fit_transform(df['Churn'])

# State has 50 different values so we remove it to keep things simple
df = df.drop('State', axis=1)

print("After converting text to numbers:")
print(df[['International plan', 'Voice mail plan', 'Churn']].head(10))

After converting text to numbers:
   International plan  Voice mail plan  Churn
0                   0                1      0
1                   0                1      0
2                   0                0      0
3                   1                0      0
4                   1                0      0
5                   1                0      0
6                   0                1      0
7                   1                0      0
8                   1                1      0
9                   0                0      0


In [20]:
#Normalize the numbers
#Difference between Account length (around 100), Total day charge (around 45), Area code (around 400)
#Big differences in scale can confuse a model therefore have a scale

scaler = StandardScaler()

# Pick the number columns we want to scale
num_cols = ['Account length', 'Area code', 'Total day minutes', 
            'Total day calls', 'Total day charge', 'Total eve minutes',
            'Total eve calls', 'Total eve charge', 'Total night minutes', 
            'Total night calls', 'Total night charge', 'Total intl minutes',
            'Total intl calls', 'Total intl charge', 'Customer service calls']

# Apply scaling to those columns
df[num_cols] = scaler.fit_transform(df[num_cols])

print("After scaling - first 5 rows:")
print(df[num_cols].head())

After scaling - first 5 rows:
   Account length  Area code  Total day minutes  Total day calls  \
0        0.699642  -0.527811           1.611746         0.484868   
1        0.158576  -0.527811          -0.334212         1.135375   
2        0.931527  -0.527811           1.203753         0.685024   
3       -0.434019  -0.692467           2.256638        -1.466653   
4       -0.665904  -0.527811          -0.238324         0.634985   

   Total day charge  Total eve minutes  Total eve calls  Total eve charge  \
0          1.579942          -0.058619        -0.050781         -0.058445   
1         -0.330194          -0.095916         0.147654         -0.095397   
2          1.179465          -1.554439         0.494917         -1.554963   
3          2.212675          -2.718509        -0.596479         -2.718922   
4         -0.235772          -1.022461         1.090224         -1.021482   

   Total night minutes  Total night calls  Total night charge  \
0             0.857403          -

In [21]:
#Split the data into training and testing sets
#Use 80% of the data to teach the model
#Use remaining 20% to test if it actually learned anything

# X is everything except what we want to predict (Churn)
X = df.drop('Churn', axis=1)

# y is what we want to predict - will a customer leave or not
y = df['Churn']

# Split the data - 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)
print()
print("We train on", len(X_train), "customers")
print("We test on", len(X_test), "customers")

Training set size: (2132, 18)
Testing set size: (534, 18)

We train on 2132 customers
We test on 534 customers
